# OzFish: fine-tune the pretrained CFD RF-DETR Medium on reef fish - Colab T4, ~1.5-2 h

Split **by camera drop** (left/right stereo views together): **50% train / 20% val / 30% test**.
- train -> fine-tuning; val -> best checkpoint and confidence threshold; test -> reported once.
- The pretrained model **as-is** is scored on the **same test drops**, so before/after is a fair comparison.
- Main metric: greedy matching at **IoU >= 0.5** (standard). Also reported, clearly labelled: **IoU >= 0.3**
  (a looser rule often used for crowded scenes with rough crowd-sourced boxes). The threshold for each is chosen on val.

OzFish: AIMS, UWA & Curtin University (2019), doi 10.25845/5e28f062c5097, CC-BY 3.0 AU. Not in CFD's training data.
**Bring back:** `MyDrive/fish/ozfish_finetune_results.json` (and `MyDrive/fish/cfd_rfdetr_ozfish_ft.pth` if it wins).

In [ ]:
!nvidia-smi -L
!pip install -q rfdetr supervision ultralytics
!wget -q -nc -O /content/cfd_rfdetr_medium.pth https://github.com/filippovarini/community-fish-detector/releases/download/2026.07.06-release/cfd-rf-detr-medium-1024-2026.03.24.cp-011.20260706-release.pth

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, hashlib, collections, requests
import numpy as np
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
DRIVE_DIR = '/content/drive/MyDrive/fish'; os.makedirs(DRIVE_DIR, exist_ok=True)
OZ = Path('/content/ozfish'); (OZ / 'frames').mkdir(parents=True, exist_ok=True)
API = 'https://storage.pawsey.org.au/api'
TOKEN = requests.post(f'{API}/login/public', timeout=60).json()['token']   # public, no login

def urls_for(paths):
    r = requests.post(f'{API}/m/public/download/prepare', headers={'X-Auth-Token': TOKEN},
                      json={'filePaths': paths}, timeout=120).json()
    assert r.get('success'), r
    return {x['filePath']: x['url'] for x in r['result']}

def fetch(url, dest, tries=4):
    for t in range(tries):
        try:
            data = requests.get(url, timeout=180).content
            if len(data) > 1000:
                dest.write_bytes(data); return True
        except Exception:
            pass
    return False

# manifests (bounding boxes)
man = [f'/FDFML/labelled/manifests/batch0{i}.json' for i in (1, 2, 3)]
for p, u in urls_for(man).items():
    fetch(u, OZ / Path(p).name)
RECS = []
for i in (1, 2, 3):
    for line in (OZ / f'batch0{i}.json').read_text().splitlines():
        if line.strip():
            r = json.loads(line)
            ann = next(v for k, v in r.items() if isinstance(v, dict) and 'annotations' in v)
            RECS.append((r['source-ref'], ann['annotations']))
print(len(RECS), 'frames,', sum(len(a) for _, a in RECS), 'boxes')

In [ ]:
need = [n for n, _ in RECS if not (OZ / 'frames' / n).exists()]
print('downloading', len(need), 'frames')
ok = 0
for i in range(0, len(need), 50):
    chunk = need[i:i + 50]
    urls = urls_for([f'/FDFML/frames/{n}' for n in chunk])
    with ThreadPoolExecutor(16) as ex:
        ok += sum(ex.map(lambda n: fetch(urls[f'/FDFML/frames/{n}'], OZ / 'frames' / n), chunk))
    print(i + len(chunk), '/', len(need))
RECS = [(n, a) for n, a in RECS if (OZ / 'frames' / n).exists()]
print('frames available:', len(RECS))

In [ ]:
drop = lambda n: re.match(r'([A-Za-z]+\d+)', n).group(1)          # camera drop, L/R views together
def split(n):
    h = int(hashlib.md5(('ft:' + drop(n)).encode()).hexdigest(), 16) % 10
    return 'train' if h < 5 else ('val' if h < 7 else 'test')
GT = {n: np.array([[a['left'], a['top'], a['left'] + a['width'], a['top'] + a['height']] for a in ann], np.float32).reshape(-1, 4)
      for n, ann in RECS}
SPLITS = {s: sorted(n for n in GT if split(n) == s) for s in ('train', 'val', 'test')}
for s, ns in SPLITS.items():
    print(s, len(ns), 'frames', len({drop(n) for n in ns}), 'drops', sum(len(GT[n]) for n in ns), 'fish')

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match_flags(gt, boxes, confs, thr=0.5):
    order = np.argsort(-confs)
    ious = iou(boxes, gt) if len(boxes) and len(gt) else np.zeros((len(boxes), len(gt)))
    used, out = set(), []
    for i in order:
        cand = [(ious[i, g], g) for g in range(len(gt)) if g not in used and ious[i, g] >= thr]
        if cand:
            used.add(max(cand)[1])
        out.append((float(confs[i]), bool(cand)))
    return out

def counts(gt, boxes, confs, conf, thr=0.5):
    k = confs >= conf
    f = match_flags(gt, boxes[k], confs[k], thr); tp = sum(x for _, x in f)
    return tp, len(f) - tp, len(gt) - tp

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def evaluate(name, preds, thr=0.5):
    tot = lambda ns, c: tuple(sum(v) for v in zip(*(counts(GT[n], *preds[n], c, thr) for n in ns)))
    table = [(prf(*tot(SPLITS['val'], c))[2], float(c)) for c in np.round(np.arange(0.05, 0.91, 0.05), 2)]
    vf1, conf = max(table)
    ns = SPLITS['test']
    per = {n: counts(GT[n], *preds[n], conf, thr) for n in ns}
    tp, fp, fn = (sum(v[k] for v in per.values()) for k in range(3)); p, r, f1 = prf(tp, fp, fn)
    groups = collections.defaultdict(list)
    for n in ns:
        groups[drop(n)].append(n)
    rng = np.random.default_rng(0); keys = list(groups); boot = []
    for _ in range(1000):
        s = [n for k in rng.choice(keys, len(keys)) for n in groups[k]]
        boot.append(prf(*(sum(per[n][k] for n in s) for k in range(3))))
    ci = lambda j: [round(float(np.percentile([b[j] for b in boot], q)), 3) for q in (2.5, 97.5)]
    flags = sorted((f for n in ns for f in match_flags(GT[n], *preds[n], thr)), key=lambda t: -t[0])
    n_gt = sum(len(GT[n]) for n in ns)
    tps = np.cumsum([f for _, f in flags]); fps = np.cumsum([not f for _, f in flags])
    mrec = np.concatenate([[0], tps / n_gt, [1]]); mpre = np.concatenate([[1], tps / np.maximum(tps + fps, 1), [0]])
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    ap50 = float(np.sum((mrec[1:] - mrec[:-1]) * mpre[1:]))
    res = {'iou': thr, 'conf_chosen_on_val': conf, 'val_f1': round(vf1, 3), 'precision': round(p, 3), 'recall': round(r, 3),
           'f1': round(f1, 3), 'ap50': round(ap50, 3), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn),
           'precision_ci': ci(0), 'recall_ci': ci(1), 'f1_ci': ci(2)}
    print(f'{name:28s} IoU {thr} conf {conf:.2f} | TEST P {p:.3f} R {r:.3f} F1 {f1:.3f} {ci(2)} AP50 {ap50:.3f}')
    return res

ALL = SPLITS['val'] + SPLITS['test']
RESULTS = {'dataset': 'OzFish bounding boxes (1,758 frames, 43,572 boxes); split by camera drop 50% train / 20% val / 30% test',
           'test_frames': len(SPLITS['test']), 'test_drops': len({drop(n) for n in SPLITS['test']}),
           'test_fish': int(sum(len(GT[n]) for n in SPLITS['test'])), 'matching': 'greedy IoU>=0.5', 'models': {}}

In [ ]:
from rfdetr import from_checkpoint
def rfdetr_preds(model):
    P = {}
    for i, n in enumerate(ALL):
        d = model.predict(Image.open(OZ / 'frames' / n).convert('RGB'), threshold=0.02)
        P[n] = (np.asarray(d.xyxy, np.float32).reshape(-1, 4), np.asarray(d.confidence, np.float32).reshape(-1))
        if i % 300 == 0: print(i, '/', len(ALL))
    return P

# 1) pretrained as-is, on the same val/test drops
cfd = from_checkpoint('/content/cfd_rfdetr_medium.pth')
P_ASIS = rfdetr_preds(cfd)
RESULTS['models']['CFD RF-DETR Medium, pretrained as-is'] = {f'iou_{t}': evaluate('as-is', P_ASIS, t) for t in (0.5, 0.3)}
json.dump(RESULTS, open(f'{DRIVE_DIR}/ozfish_finetune_results.json', 'w'), indent=2)
del cfd

In [ ]:
# 2) COCO-format dataset for RF-DETR (train / valid / test folders; images linked, not copied)
import shutil
DS = Path('/content/ozfish_coco')
shutil.rmtree(DS, ignore_errors=True)
for s_in, s_out in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
    (DS / s_out).mkdir(parents=True)
    images, anns = [], []
    for i, n in enumerate(SPLITS[s_in]):
        os.symlink(OZ / 'frames' / n, DS / s_out / n)
        w, h = Image.open(OZ / 'frames' / n).size
        images.append({'id': i, 'file_name': n, 'width': w, 'height': h})
        for b in GT[n]:
            x0, y0 = max(0.0, float(b[0])), max(0.0, float(b[1]))
            x1, y1 = min(float(w), float(b[2])), min(float(h), float(b[3]))
            if x1 - x0 > 1 and y1 - y0 > 1:
                anns.append({'id': len(anns), 'image_id': i, 'category_id': 0, 'bbox': [x0, y0, x1 - x0, y1 - y0],
                             'area': (x1 - x0) * (y1 - y0), 'iscrowd': 0})
    json.dump({'images': images, 'annotations': anns, 'categories': [{'id': 0, 'name': 'fish', 'supercategory': 'none'}]},
              open(DS / s_out / '_annotations.coco.json', 'w'))
    print(s_out, len(images), 'images', len(anns), 'boxes')

In [ ]:
# 3) fine-tune (low learning rate: the model already knows fish). Roughly 5-7 min per epoch on a T4.
#    If CUDA runs out of memory: batch_size=1, grad_accum_steps=16.
FT_EPOCHS = 12
OUT_FT = '/content/rfdetr_ozfish_ft'
cfd = from_checkpoint('/content/cfd_rfdetr_medium.pth')
cfd.train(dataset_dir=str(DS), output_dir=OUT_FT, epochs=FT_EPOCHS, batch_size=2, grad_accum_steps=8,
          lr=5e-5, lr_encoder=5e-5, warmup_epochs=1.0, checkpoint_interval=1)

In [ ]:
# 4) score the fine-tuned model (best checkpoint chosen by RF-DETR on the valid split)
import glob
ck = [p for name in ('checkpoint_best_total.pth', 'checkpoint_best_ema.pth', 'checkpoint_best_regular.pth')
      for p in glob.glob(f'{OUT_FT}/**/{name}', recursive=True)]
ck = ck or sorted(glob.glob(f'{OUT_FT}/**/*.pth', recursive=True), key=os.path.getmtime)[-1:]
print('using', ck[0])
ft = from_checkpoint(ck[0])
P_FT = rfdetr_preds(ft)
key = 'CFD RF-DETR Medium, fine-tuned on OzFish train'
RESULTS['models'][key] = {f'iou_{t}': evaluate('fine-tuned', P_FT, t) for t in (0.5, 0.3)}
RESULTS['models'][key]['checkpoint'] = ck[0]

# keep the one with higher VAL F1 at IoU 0.5 (test is never used to choose)
a = RESULTS['models']['CFD RF-DETR Medium, pretrained as-is']['iou_0.5']
b = RESULTS['models'][key]['iou_0.5']
RESULTS['kept'] = 'fine-tuned' if b['val_f1'] > a['val_f1'] else 'pretrained as-is'
if RESULTS['kept'] == 'fine-tuned':
    shutil.copyfile(ck[0], f'{DRIVE_DIR}/cfd_rfdetr_ozfish_ft.pth')
json.dump(RESULTS, open(f'{DRIVE_DIR}/ozfish_finetune_results.json', 'w'), indent=2)
print(json.dumps(RESULTS, indent=1))
print('KEPT (by val F1 at IoU 0.5):', RESULTS['kept'])